# Observation and action boundary tests

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import unittest,tempfile,json
from pathlib import Path
import numpy as np
from environment_adapter import BaselineState
from baseline_rewards import TaskReward,MaxSupportReward
from types import SimpleNamespace
import gymnasium as gym
from environment_adapter import MatchedBaseline
class NativeFixture(gym.Env):
    def __init__(self):
        self.action_space=gym.spaces.MultiDiscrete([3,3,4]);self.observation_space=gym.spaces.Box(-np.inf,np.inf,(86,),dtype=np.float32)
        self.env=SimpleNamespace(game='Solid',classifier=True,preference=True,imitation_learning=False,discretize=False,decision_period=10,cluster=0,period_ra=False,weight=.5,model=None)
class Checks(unittest.TestCase):
    def test_observation_shape_and_nonfinite(self):
        s=BaselineState(lambda p:(.5,{}),'TASK');obs=s.reset(np.zeros(81))
        self.assertEqual(obs.shape,(86,));self.assertEqual(obs.dtype,np.float32)
        with self.assertRaises(ValueError):s.step(np.zeros(80),0)
        with self.assertRaises(ValueError):s.step(np.full(81,np.nan),0)
    def test_real_adapter_action_guard(self):
        from unittest.mock import patch
        with patch('environment_adapter.FrozenPreferenceScore',autospec=True) as score:
            score.return_value.assert_frozen.return_value=None
            adapter=MatchedBaseline(NativeFixture(),'TASK')
        self.assertTrue(adapter.action_space.contains(np.array([2,2])))
        for action in [np.array([3,0]),np.array([0,-1]),np.array([0,0,1])]:
            with self.assertRaises(ValueError):adapter.step(action)
print('Observation and action boundary tests definitions/execution completed.')


Frozen increasing-preference scoring definitions/execution completed.
Task progress and fresh-pair rewards definitions/execution completed.
Matched observation action and window adapter definitions/execution completed.
Observation and action boundary tests definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Tests run:',result.testsRun)

test_observation_shape_and_nonfinite (__main__.Checks) ... 

ok


test_real_adapter_action_guard (__main__.Checks) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.005s

OK


Tests run: 2
